# Basic RAG

Basic RAG with Langchain

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 09/03/2026   | Martin | Created   | Started basic RAG application | 

# Content

* [Introduction](#introduction)

# Introduction

This demonstrates the fundamental RAG pattern:
  1. Load documents
  2. Split into chunks
  3. Embed and store in ChromaDB
  4. Retrieve relevant chunks on query
  5. Generate answer with LLM

ARCHITECTURAL DECISIONS:
- RecursiveCharacterTextSplitter: Tries to split on natural boundaries
  (paragraphs → sentences → words) before resorting to character splits.
  This preserves semantic coherence within chunks.

- chunk_size=500, chunk_overlap=50: A starting heuristic. Too small = 
  missing context; too large = noisy retrieval + hitting context limits.
  Overlap ensures sentences split across boundaries aren't lost.

- HuggingFaceEmbeddings (all-MiniLM-L6-v2): A lightweight, fast model
  that runs locally without API costs. For production, consider 
  text-embedding-ada-002 (OpenAI) or text-embedding-3-small for better
  quality at scale.

- ChromaDB (in-memory): Zero-setup vector store, ideal for development.
  In production you'd persist to disk or use a hosted solution.

- k=3 retrieval: Fetching top-3 chunks balances context richness vs.
  noise. Too many chunks overwhelm the LLM; too few miss relevant info.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate

ModuleNotFoundError: No module named 'langchain.chains'

In [4]:
# ==============================
# Document Loading
# ==============================
def load_documents(data_dir: str = "./data"):
  """
  Loads text files from a directory. Extend this with different loaders for
  PDFs, webpages, databases, and other knowledge sources
  """
  loader = DirectoryLoader(
    path=data_dir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
  )
  docs = loader.load()
  print(f"[LOAD] Loaded {len(docs)} documents from '{data_dir}'")

  return docs

In [5]:
docs = load_documents()

100%|██████████| 3/3 [00:00<00:00, 174.02it/s]

[LOAD] Loaded 3 documents from './data'


Chunking helps to retrieve only the relevant pieces from a document. Wrong chunk size = wrong answers.

In [10]:
# ==============================
# Chunking
# ==============================
def chunk_documents(docs, chunk: int = 500, chunk_overlap: int = 50):
  """
  RecursiveCharacterTextSplitter priority order:
  ["\\n\\n", "\\n", " ", ""] — paragraph → line → word → char
  """
  splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk,
    chunk_overlap=chunk_overlap,
    length_function=len,
    add_start_index=True # Stores char offset in metadata: useful for highlighting source passage
  )
  chunks = splitter.split_documents(docs)
  print(f"[CHUNK] Split into {len(chunks)} chunks (size={chunk}, overlap={chunk_overlap})")

  return chunks

In [11]:
chunks = chunk_documents(docs, chunk=500, chunk_overlap=50)
chunks

[CHUNK] Split into 21 chunks (size=500, overlap=50)


[Document(metadata={'source': 'data\\llm_and_rag.txt', 'start_index': 0}, page_content='Large Language Models and AI Systems\n\nTransformer Architecture\nThe Transformer architecture, introduced in "Attention is All You Need" (2017), uses self-attention mechanisms to process sequences in parallel. Key components include multi-head attention, positional encodings, and feed-forward layers. Unlike RNNs, transformers can capture long-range dependencies efficiently through the attention mechanism.'),
 Document(metadata={'source': 'data\\llm_and_rag.txt', 'start_index': 408}, page_content='Attention Mechanism\nSelf-attention computes relationships between all tokens in a sequence simultaneously. Query, Key, and Value matrices are computed from input embeddings. The attention score between positions is: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V. Multi-head attention runs several attention operations in parallel, capturing different types of relationships.'),
 Document(metadata={'source': 

Using embeddings are better than keyword searches because it allows the similarity search to be performed semantically, especially good for conceptual questions

In [15]:
# ==============================
# Embedding + Vector Store
# ==============================
def build_vectorstore(chunks, persist_dir: str = None):
  """
  Embeddings are an expensive step. Pre-compute and persist in vectorstore
  in Production

  all-MiniLM-L6-v2 is a sentence transformer
  """
  print("[EMBED] Loading embedding model...")
  embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
  )

  print("[STORE] Building ChromaDB vectorstore...")
  kwargs = {"documents": chunks, "embedding": embeddings}
  if persist_dir:
    kwargs["persist_directory"] = persist_dir
  store = Chroma.from_documents(**kwargs)
  print(f"[STORE] Stored {store._collection.count()} vectors in ChromaDB")
  return store, embeddings


In [17]:
store = build_vectorstore(chunks)
store

[EMBED] Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12636.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[STORE] Building ChromaDB vectorstore...
[STORE] Stored 42 vectors in ChromaDB


(<langchain_community.vectorstores.chroma.Chroma at 0x1e4bcf6f020>,
 HuggingFaceEmbeddings(client=SentenceTransformer(
   (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
   (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
   (2): Normalize()
 ), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, multi_process=False, show_progress=False))

Retrieval converts a query into an embedding and finds the k most similar chunks via cosine similarity (or wtv similarity metric chosen)

In [ ]:
# ==============================
# Retrieval
# ==============================
def build_retriever(store, k: int = 3):
  """
  k=3 is a reasonable default. Production depends on:
  - Average answer complexity
  - LLM context window
  - Latency requirements
  """
  retriever = store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": k}
  )

  return retriever

In [ ]:
RAG_PROMPT = PromptTemplate(
  input_variables=["context", "question"],
  template="""
  You are a helpful assistant. Use ONLY the following context to answer the question.
  If the answer isn't in the context, say "I don't know based on the provided documents."ArithmeticError
  
  Context:
  {context}

  Question: {question}

  Answer:
  """
)

In [ ]:
%load_ext watermark
%watermark